#RESTART:
- new dataset from kaggle https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-dataset?resource=download

In [1]:
# ===================================
# CELL 1: DOWNLOAD (SKIP IF EXISTS)
# ===================================

from google.colab import drive
import pandas as pd
import shutil
import os
import json
import time

drive.mount('/content/drive')

!pip install -q kaggle

os.makedirs('/root/.kaggle', exist_ok=True)

kaggle_credentials = {
    "username": "biancagambino",
    "key": "4cfe6253c2484adad8e890df33cdf855"
}

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

!chmod 600 /root/.kaggle/kaggle.json

print("✅ Kaggle credentials set up")

# Check if already downloaded
if os.path.exists('fashion-product-images-dataset.zip'):
    file_size = os.path.getsize('fashion-product-images-dataset.zip')
    print(f"\n✅ Zip already exists ({file_size / (1024*1024):.1f} MB)")
    print("Skipping download...")
else:
    print("\n📥 Downloading dataset...")
    !wget -O fashion-product-images-dataset.zip https://www.kaggle.com/api/v1/datasets/download/paramaggarwal/fashion-product-images-dataset

    if os.path.exists('fashion-product-images-dataset.zip'):
        print("✅ Download complete!")
    else:
        print("❌ Download failed")

print("\n✅ Ready for extraction (run Cell 2)")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ===================================
# EXTRACT AND COPY - CELL 2
# ===================================

import pandas as pd
import shutil
import os
import glob

# Remove any previously extracted files
print("🧹 Removing old extracted files...")
!rm -rf fashion-dataset styles.csv images.csv images

print("📦 Extracting dataset (5-10 minutes)...")
# -o forces overwrite without asking
!unzip -o -q fashion-product-images-dataset.zip

print("✅ Extract complete!")

# Find files
print("\n🔍 Finding dataset files...")
csv_files = glob.glob('**/styles.csv', recursive=True)
if not csv_files and os.path.exists('styles.csv'):
    csv_files = ['styles.csv']

if csv_files:
    styles_path = csv_files[0]
    print(f"✅ Found: {styles_path}")

    dataset_dir = os.path.dirname(styles_path) or '.'
    images_dir = os.path.join(dataset_dir, 'images')

    if not os.path.exists(images_dir):
        image_dirs = glob.glob('**/images', recursive=True)
        if image_dirs:
            images_dir = image_dirs[0]

    print(f"✅ Images: {images_dir}")
    num_images = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
    print(f"   Total: {num_images} images")

    # Load and sample
    print("\n📊 Processing...")
    df = pd.read_csv(styles_path, on_bad_lines='skip')
    print(f"Loaded {len(df)} rows")

    df_sampled = df.groupby('articleType', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 200), random_state=42)
    )
    if len(df_sampled) > 5000:
        df_sampled = df_sampled.sample(n=5000, random_state=42)

    print(f"Sampled {len(df_sampled)} images")

    # Copy to Drive
    print("\n📤 Copying to Google Drive...")
    base_path = '/content/drive/MyDrive/ClosetGeniusCNN'
    new_images_path = f'{base_path}/images'

    if os.path.exists(new_images_path):
        shutil.rmtree(new_images_path)
    os.makedirs(new_images_path, exist_ok=True)

    copied = 0
    for idx, row in df_sampled.iterrows():
        src = os.path.join(images_dir, f"{row['id']}.jpg")
        dst = os.path.join(new_images_path, f"{row['id']}.jpg")
        if os.path.exists(src):
            shutil.copy(src, dst)
            copied += 1
            if copied % 500 == 0:
                print(f"  ✓ {copied} images...")

    print(f"\n✅ Copied {copied} images")

    df_sampled.to_csv(f'{base_path}/styles.csv', index=False)

    images_csv = os.path.join(dataset_dir, 'images.csv')
    if os.path.exists(images_csv):
        shutil.copy(images_csv, f'{base_path}/images.csv')

    print(f"\n🎉 DONE!")
    print(f"📁 {base_path}")
    print(f"🖼️  {copied} images")
    print(f"\n✅ READY TO TRAIN!")

🧹 Removing old extracted files...
📦 Extracting dataset (5-10 minutes)...
✅ Extract complete!

🔍 Finding dataset files...
✅ Found: drive/MyDrive/closetgenius1/Data1/fashion-dataset/styles.csv
✅ Images: drive/MyDrive/closetgenius1/Data1/fashion-dataset/images
   Total: 1393 images

📊 Processing...
Loaded 44424 rows
Sampled 5000 images

📤 Copying to Google Drive...


/tmp/ipython-input-438585562.py:47: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled = df.groupby('articleType', group_keys=False).apply(



✅ Copied 156 images

🎉 DONE!
📁 /content/drive/MyDrive/ClosetGeniusCNN
🖼️  156 images

✅ READY TO TRAIN!


In [ ]:
# ===================================
# FLORENCE-2: INSTALL DEPENDENCIES
# Run this cell once per Colab session
# ===================================

!pip install transformers==4.43.0 timm einops -q


In [ ]:
# ===================================
# FLASH_ATTN MOCK
# Florence-2 checks for flash_attn before loading.
# This mocks it so the check passes — eager attention is used instead.
# ===================================

import sys
import types
import importlib.machinery

mock_flash = types.ModuleType('flash_attn')
mock_flash.__spec__ = importlib.machinery.ModuleSpec('flash_attn', None)
sys.modules['flash_attn'] = mock_flash
sys.modules['flash_attn.flash_attn_interface'] = mock_flash
sys.modules['flash_attn.bert_padding'] = mock_flash
sys.modules['flash_attn.flash_attn_utils'] = mock_flash

print('flash_attn mocked')


In [ ]:
# ===================================
# FLORENCE-2: LOAD MODEL
# Uses florence-2-base (~270M params) to fit Colab GPU memory.
# Swap to 'microsoft/Florence-2-large' on an A100 for best quality.
# ===================================

import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image

FLORENCE_MODEL_ID = 'microsoft/Florence-2-base'
florence_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
florence_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'Loading Florence-2 on {florence_device} ({florence_dtype})...')

florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID,
    torch_dtype=florence_dtype,
    trust_remote_code=True,
    attn_implementation='eager'
)

florence_model.config.forced_bos_token_id = None
florence_model = florence_model.to(florence_device)

florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID,
    trust_remote_code=True
)

florence_model.eval()
print('Florence-2 loaded')


In [ ]:
# ===================================
# FLORENCE-2: CLOTHING ANALYZER CLASS
# ===================================

import re

class FlorenceClothingAnalyzer:
    def __init__(self, model, processor, device, dtype):
        self.model = model
        self.processor = processor
        self.device = device
        self.dtype = dtype

    def _run_task(self, image, task_prompt, text_input=''):
        prompt = task_prompt + text_input
        inputs = self.processor(text=prompt, images=image, return_tensors='pt')
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        if 'pixel_values' in inputs:
            inputs['pixel_values'] = inputs['pixel_values'].to(self.dtype)
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, max_new_tokens=128, do_sample=False)
        output_text = self.processor.batch_decode(output_ids, skip_special_tokens=False)[0]
        parsed = self.processor.post_process_generation(
            output_text, task=task_prompt, image_size=(image.width, image.height)
        )
        return parsed.get(task_prompt, '')

    def describe(self, image):
        caption = self._run_task(image, '<CAPTION>')
        detailed = self._run_task(image, '<DETAILED_CAPTION>')
        words = re.findall(r'\b[a-zA-Z]{3,}\b', detailed.lower())
        fashion_keywords = {
            'shirt','blouse','dress','pants','jeans','jacket','coat',
            'skirt','sweater','hoodie','shorts','suit','vest','top',
            'red','blue','green','black','white','gray','grey','pink',
            'yellow','orange','purple','brown','navy','beige','cream',
            'striped','floral','plaid','solid','printed','patterned',
            'casual','formal','sporty','elegant','slim','oversized',
            'cotton','denim','wool','silk','polyester','linen',
            'long','short','sleeveless','cropped','fitted','loose'
        }
        tags = list(set(w for w in words if w in fashion_keywords))
        return {'caption': caption, 'detailed_caption': detailed, 'tags': tags}


florence_analyzer = FlorenceClothingAnalyzer(
    florence_model, florence_processor, florence_device, florence_dtype
)
print('FlorenceClothingAnalyzer ready')


# ===================================
# LABEL EXTRACTION
# Maps Florence captions/tags to structured clothing labels.
# Used by the Flask /scan endpoint.
# ===================================

CATEGORY_MAP = {
    't-shirt': 'Tshirts',   'tshirt': 'Tshirts',    'tee': 'Tshirts',
    'shirt': 'Shirts',      'blouse': 'Tops',        'top': 'Tops',
    'tank': 'Tops',         'camisole': 'Tops',
    'dress': 'Dresses',     'gown': 'Dresses',       'maxi': 'Dresses',
    'skirt': 'Skirts',      'midi': 'Skirts',
    'jeans': 'Jeans',       'denim': 'Jeans',
    'pants': 'Trousers',    'trousers': 'Trousers',  'chinos': 'Trousers',
    'shorts': 'Shorts',     'leggings': 'Leggings',
    'joggers': 'Track Pants','sweatpants': 'Track Pants',
    'jacket': 'Jackets',    'coat': 'Jackets',       'blazer': 'Blazers',
    'hoodie': 'Sweatshirts','sweatshirt': 'Sweatshirts',
    'sweater': 'Sweaters',  'pullover': 'Sweaters',  'cardigan': 'Sweaters',
    'vest': 'Waistcoat',    'waistcoat': 'Waistcoat',
    'sneakers': 'Sports Shoes', 'sneaker': 'Sports Shoes',
    'shoes': 'Casual Shoes','loafers': 'Casual Shoes','boots': 'Casual Shoes',
    'heels': 'Heels',       'pumps': 'Heels',
    'sandals': 'Sandals',   'flip flops': 'Flip Flops','flats': 'Casual Shoes',
    'watch': 'Watches',     'bag': 'Handbags',       'handbag': 'Handbags',
    'purse': 'Handbags',    'backpack': 'Backpacks',
    'belt': 'Belts',        'tie': 'Ties',
    'hat': 'Caps',          'cap': 'Caps',            'beanie': 'Caps',
    'sunglasses': 'Sunglasses',
    'earrings': 'Earrings', 'necklace': 'Necklace and Chains',
    'scarf': 'Scarves',     'socks': 'Socks',
    'kurta': 'Kurtas',      'saree': 'Sarees',
    'jumpsuit': 'Jumpsuit', 'tracksuit': 'Tracksuits',
    'swimsuit': 'Swimwear', 'bikini': 'Swimwear',
}

COLOUR_MAP = {
    'black': 'Black',       'white': 'White',
    'blue': 'Blue',         'navy': 'Navy Blue',
    'red': 'Red',           'maroon': 'Maroon',      'burgundy': 'Maroon',
    'green': 'Green',       'olive': 'Green',
    'yellow': 'Yellow',     'mustard': 'Yellow',
    'orange': 'Orange',
    'purple': 'Purple',     'violet': 'Purple',      'lavender': 'Lavender',
    'pink': 'Pink',         'rose': 'Pink',           'coral': 'Pink',
    'brown': 'Brown',       'tan': 'Brown',           'camel': 'Brown',
    'grey': 'Grey',         'gray': 'Grey',           'charcoal': 'Grey',
    'beige': 'Beige',       'cream': 'Cream',
    'gold': 'Gold',         'silver': 'Silver',
    'teal': 'Teal',         'turquoise': 'Teal',
    'khaki': 'Khaki',
}

SEASON_MAP = {
    'summer': 'Summer',     'lightweight': 'Summer', 'sleeveless': 'Summer',
    'winter': 'Winter',     'wool': 'Winter',         'knit': 'Winter',
    'fall': 'Fall',         'autumn': 'Fall',
    'spring': 'Spring',     'trench': 'Spring',
    'coat': 'Winter',       'puffer': 'Winter',
    'shorts': 'Summer',     'hoodie': 'Fall',
}

USAGE_MAP = {
    'casual': 'Casual',     'everyday': 'Casual',
    'formal': 'Formal',     'business': 'Formal',    'office': 'Formal',
    'sports': 'Sports',     'athletic': 'Sports',    'gym': 'Sports',
    'workout': 'Sports',    'activewear': 'Sports',
    'ethnic': 'Ethnic',     'traditional': 'Ethnic',
    'party': 'Party',       'evening': 'Party',
    'smart casual': 'Smart Casual',
    'travel': 'Travel',
    'home': 'Home',         'lounge': 'Home',
}


def extract_florence_labels(florence_result):
    tags = [t.lower() for t in florence_result.get('tags', [])]
    all_text = (florence_result.get('detailed_caption', '') + ' ' +
                florence_result.get('caption', '')).lower()
    words = set(tags) | set(re.findall(r'\b[a-zA-Z]{3,}\b', all_text))
    word_list = re.findall(r'\b[a-zA-Z]{3,}\b', all_text)
    bigrams = {f'{word_list[i]} {word_list[i+1]}' for i in range(len(word_list) - 1)}

    def first_match(mapping):
        for phrase in bigrams:
            if phrase in mapping:
                return mapping[phrase]
        for word in words:
            if word in mapping:
                return mapping[word]
        return None

    return {
        'category': first_match(CATEGORY_MAP) or 'Tops',
        'color':    first_match(COLOUR_MAP)   or 'Black',
        'season':   first_match(SEASON_MAP)   or 'Summer',
        'usage':    first_match(USAGE_MAP)    or 'Casual',
    }


print('Label extraction ready')


In [ ]:
# ===================================
# GROQ: INSTALL
# ===================================

!pip install -q groq
print("Groq installed.")

In [ ]:
# ===================================
# GROQ: CLIENT + CHAT FUNCTION
# ===================================

from groq import Groq

GROQ_API_KEY = "3BPvoQau3e56kiFNeCc9VYOsBPB_5GvKDsvoXd8d4gYVSBm9s"
# or: from google.colab import userdata; GROQ_API_KEY = userdata.get("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL_ID = "llama-3.3-70b-versatile"

def build_system_prompt(closet_items):
    if not closet_items:
        desc = "The user's closet is currently empty."
    else:
        lines = []
        for item in closet_items:
            parts = [item.get("name",""), f"({item.get('category','')})",
                     item.get("color",""), item.get("style",""), item.get("formality","")]
            if item.get("notes"): parts.append(f'\u2014 "{item["notes"]}"')
            if item.get("custom_tags"): parts.append(f'[{", ".join(item["custom_tags"])}]')
            lines.append(" | ".join(p for p in parts if p))
        desc = "\n".join(f"- {l}" for l in lines)
    prompt = "You are ClosetGenius Assistant, a friendly personal stylist AI.\n"
    prompt += "USER'S CLOSET:\n" + desc + "\n"
    prompt += "Suggest specific items from the closet by name. Group into complete outfits.\n"
    prompt += "Keep responses concise and friendly (under 250 words). Never invent items."
    return prompt

def chat_with_llama(conversation_history, closet_items):
    messages = [{"role": "system", "content": build_system_prompt(closet_items)}] + conversation_history
    response = groq_client.chat.completions.create(
        model=MODEL_ID, messages=messages, max_tokens=512, temperature=0.7)
    return response.choices[0].message.content

print("Groq client ready:", MODEL_ID)

In [ ]:
# ===================================
# FLASK API — FLORENCE-ONLY
# /scan uses Florence-2 directly — no AlexNet.
# Florence handles category, colour, season, usage + description.
# ===================================

!pip install -q flask pyngrok

from flask import Flask, request, jsonify
from pyngrok import ngrok
import tempfile, os

app = Flask(__name__)


@app.route('/scan', methods=['POST'])
def scan():
    if 'image' not in request.files:
        return jsonify({'error': 'No image uploaded'}), 400

    file = request.files['image']
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        file.save(tmp.name)
        tmp_path = tmp.name

    try:
        image = Image.open(tmp_path).convert('RGB')
        florence_result = florence_analyzer.describe(image)
        labels = extract_florence_labels(florence_result)

        return jsonify({
            'category':    labels['category'],
            'color':       labels['color'],
            'season':      labels['season'],
            'usage':       labels['usage'],
            'description': florence_result.get('detailed_caption', ''),
            'tags':        florence_result.get('tags', []),
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500
    finally:
        os.unlink(tmp_path)


@app.route('/chat', methods=['POST'])
def chat():
    try:
        body = request.get_json(force=True)
    except Exception:
        return jsonify({'error': 'Invalid JSON body'}), 400
    user_message = body.get('message', '').strip()
    if not user_message:
        return jsonify({'error': "Missing 'message' field"}), 400
    closet_items = body.get('closet', [])
    history = body.get('history', [])
    conversation = history + [{'role': 'user', 'content': user_message}]
    try:
        reply = chat_with_llama(conversation, closet_items)
        return jsonify({'reply': reply})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'models': ['Florence-2', 'Llama-3.3-70b']})


ngrok.set_auth_token('3B3H31JuVlU1FMMqAPjN0OrTHOO_6ab3tvH9guDrarNGpGHfA')
ngrok_tunnel = ngrok.connect(5000)
print(f'\nAPI live at: {ngrok_tunnel.public_url}')
print(f'  POST {ngrok_tunnel.public_url}/scan')
print(f'  POST {ngrok_tunnel.public_url}/chat')
print(f'  GET  {ngrok_tunnel.public_url}/health')

app.run(port=5000)
